# Component 1 | U-Net V3 training evidence

Frozen trained ResNet18 U-Net. This notebook replays actual saved training observations; it never trains or changes weights. Saved cell outputs were executed locally on the development computer, **not in Google Colab**. Runtime GPU is not needed for evidence replay.

Open cells in order. Keep the private, noncommercial attribution package together. Current scientific status: bounded model/engineering evidence; no approved real Twin, safety certification, planner approval or established algorithmic novelty.


## Retained curated demo: cubi_03304

A 12-candidate search did not find a materially better full-frame result. The higher-scoring cubi_00412 was rejected as a replacement after full-size visual QA because of severe exterior artifacts. The existing simple cubi_03304 is retained, **not presented as corrected or flawless**. Its selected-example IoU is not unbiased model performance. Raw errors, every trial and cubi_03153 remain preserved; historical training/evaluation records are unchanged.


In [ ]:
from pathlib import Path
import os, sys, json
MYDRIVE_PACKAGE = "COMPONENT_1_PP1_COLAB_DELIVERY_V1"  # change if your Drive folder name differs
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount('/content/drive')  # normal user authorization, no credentials in this notebook
    PACKAGE_ROOT = Path('/content/drive/MyDrive') / MYDRIVE_PACKAGE
else:
    PACKAGE_ROOT = Path(os.environ.get('PP1_COLAB_PACKAGE', Path.cwd())).resolve()
    if PACKAGE_ROOT.name == '01_NOTEBOOKS':
        PACKAGE_ROOT = PACKAGE_ROOT.parent
assert (PACKAGE_ROOT / 'SOURCE_PROVENANCE.json').is_file(), 'Choose the extracted package root, not the ZIP or notebook folder.'
sys.path.insert(0, str(PACKAGE_ROOT / '09_RUNTIME'))
from portable_unet import read, verify_assets
def show_png(path):
    from PIL import Image
    import matplotlib.pyplot as plt
    with Image.open(path) as image:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.imshow(image)
    ax.axis('off'); fig.tight_layout(); plt.show()
print('Execution location:', 'Colab (remote run by you)' if IN_COLAB else 'Local Python; NOT a remote Colab run')
print(verify_assets(PACKAGE_ROOT))


## What was trained?

Semantic segmentation assigns a class to each pixel. A ResNet18 encoder extracts progressively coarser features; a U-Net decoder combines these with skip features to recover a dense four-class map. The original trainer loaded official ImageNet encoder weights from a local file, then trained the task decoder and encoder. Inference reconstructs that architecture with `encoder_weights=None`, then **strictly loads every saved task weight**. It never predicts using unfilled random weights.

Publisher native classes: room_region=0, wall=1, door=2, window=3. Unassigned pixels and padding=255 are ignored. There is no supervised exterior/background class. A room prediction is not an approved usable event boundary.


In [ ]:
config = read(PACKAGE_ROOT / '02_ORIGINAL_TRAINING_EVIDENCE/TRAINING_CONFIG.json')
result = read(PACKAGE_ROOT / '02_ORIGINAL_TRAINING_EVIDENCE/TRAINING_RESULT.json')
dataset = read(PACKAGE_ROOT / '03_DATASET_EVIDENCE/dataset_summary.json')
print(json.dumps(config, indent=2))
print('Recorded training status:', result['status'], '| epochs:', result['epochs'])
print('Partitions:', dataset['counts'])
print('Qualified sample scope:', dataset['limitations'])


## Training, validation and selection

Actual run: 16 epochs, batch 4, seed 20261018; AdamW lr=0.0003, weight_decay=0.0001; CosineAnnealingLR(T_max=16). Training used paired flips/90-degree rotations, brightness/contrast and occasional blur. Validation had no augmentation. RGB input uses aspect-preserving 512 letterbox then ImageNet mean/std normalization. Mask resizing is nearest-neighbour.

Loss = train-frequency capped inverse-square-root weighted cross-entropy + 0.5 × macro soft Dice on valid pixels. During original training, backward propagation and AdamW updated the parameters. **No backward pass or optimizer is invoked by this package.** The maximum pooled validation four-class mIoU selects the checkpoint; diagnostic data does not select it.

Train curves use augmented data while weights change across batches. They should not be interpreted as a directly matched frozen-model validation comparison. No observed LR/gradient history was recorded for U-Net, so none is invented.


In [ ]:
history = read(PACKAGE_ROOT / '02_ORIGINAL_TRAINING_EVIDENCE/history.json')
validation = [row for row in history if row['phase'] == 'val']
best = max(validation, key=lambda row: row['miou'])
assert len(history) == 32 and best['epoch'] == 13
assert best['miou'] == result['best_val_miou']
print('Selected epoch:', best['epoch'], '| validation mIoU:', best['miou'])
print('These are observed history values, not regenerated training.')


In [ ]:
import tempfile
from portable_unet import plot_training
plot_dir = Path(tempfile.mkdtemp(prefix='pp1-evidence-')) / 'training_curves'
plot_paths = plot_training(PACKAGE_ROOT, plot_dir)
for path in plot_paths:
    show_png(path)


## Diagnostic evaluation and domain shift

IoU_c = TP_c / (TP_c + FP_c + FN_c); mIoU is the mean over the four supported classes. The pooled confusion matrix counts only pixels whose target is not 255. Pixels ignored by the target do not penalize exterior false predictions.

Native diagnostic_test is 64 historically exposed source families, not an independent external test. All four classes have support. The ten CVC sheets below are previously observed native-label diagnostics and use different door semantics; they are not an untouched test or a controlled causal improvement estimate. The later REAL_INDOOR_ACCEPTANCE experiment uses a different ten-record selection and reports 0.56925393; do not substitute it into this earlier V3 evaluation. The old checkpoint's historical stretched CVC result 0.28178467 is another protocol and is not this V3 result.


In [ ]:
evaluation = read(PACKAGE_ROOT / '02_ORIGINAL_TRAINING_EVIDENCE/EVALUATION.json')
for domain in ['native', 'observed_cvc']:
    report = evaluation['native_v3'][domain]
    print(domain, '| records:', len(report['records']), '| pooled mIoU:', report['pooled_metrics']['miou'])
    print('Class IoUs:', dict(zip(config['classes'], report['pooled_metrics']['class_iou'])))
    print('GT support:', report['pooled_metrics']['support'])


## Genuine retained example — raw prediction and known errors

The panels show cubi_03304: original source, qualified native GT, unchanged prediction, valid-pixel disagreement and full overlay. Exterior regions are visibly incorrect/unverified. The model has no background class and the evaluation ignores unassigned pixels, so the example score does not qualify the entire sheet. No candidate in the bounded search justified a visually convincing replacement.


In [ ]:
demo_metrics=read(PACKAGE_ROOT / '06_GENUINE_SAVED_OUTPUTS/UNET_CURATED_DEMO/cubi_03304/METRICS.json')
print('CURATED selected-example IoUs, not unbiased performance:', dict(zip(['room_region','wall','door','window'],demo_metrics['room_wall_door_window_iou'])))
print('Selected-example mIoU:',demo_metrics['miou'])
show_png(PACKAGE_ROOT / '06_GENUINE_SAVED_OUTPUTS/UNET_CURATED_DEMO/cubi_03304/REVIEW_PANELS.png')


## Examiner summary

Trained model and recorded validation/diagnostic metrics are demonstrated. Technical/source-family limits, noncommercial dataset terms, poor door transfer and ignored exterior remain visible. The source drawing does not become metric or approved because it was segmented. S-GCN, weather reasoning and genuine planner-benefit validation are not implemented by these notebooks.
